In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02 — Sentiment Scoring (Run on Google Colab)\n",
    "This notebook runs FinBERT on 43,485 news headlines to produce daily company-specific sentiment scores.\n",
    "**Run this entire notebook on Google Colab with GPU enabled.**\n",
    "\n",
    "Runtime → Change runtime type → T4 GPU"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 1 — Check GPU"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "print('CUDA available:', torch.cuda.is_available())\n",
    "print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 2 — Mount Google Drive"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "\n",
    "import os\n",
    "os.makedirs('/content/drive/MyDrive/MARL/data/sentiment', exist_ok=True)\n",
    "print('Drive mounted and folders ready.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3 — Clone your GitHub repo\n",
    "Replace `yourusername` with your actual GitHub username."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!git clone https://github.com/yourusername/MARL.git\n",
    "%cd MARL\n",
    "!pip install -r requirements.txt -q"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4 — Upload news file\n",
    "Upload your `news_filtered.csv` to Colab using the cell below.\n",
    "This saves it to the correct location inside the repo."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import files\n",
    "import shutil\n",
    "\n",
    "print('Upload your news_filtered.csv file...')\n",
    "uploaded = files.upload()\n",
    "\n",
    "for filename in uploaded.keys():\n",
    "    shutil.move(filename, 'data/raw/news_filtered.csv')\n",
    "    print(f'Moved {filename} → data/raw/news_filtered.csv')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 5 — Validate news file loaded correctly"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '/content/MARL')\n",
    "\n",
    "from src.data.fetch_sentiment import load_news, load_config\n",
    "\n",
    "config = load_config()\n",
    "tickers = config['data']['tickers']\n",
    "df = load_news(tickers=tickers)\n",
    "print('\\nNews file validated successfully.')\n",
    "print(df[['date', 'ticker', 'article_title']].head(3).to_string())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 6 — Run FinBERT scoring\n", "This is the slow step — expect ~15-20 minutes on T4 GPU."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.data.fetch_sentiment import score_and_save\n",
    "score_and_save(config)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 7 — Verify output files"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import os\n",
    "\n",
    "for ticker in tickers:\n",
    "    path = f'data/sentiment/{ticker}_sentiment.csv'\n",
    "    if os.path.exists(path):\n",
    "        df = pd.read_csv(path, parse_dates=['date'], index_col='date')\n",
    "        print(f'{ticker}: {len(df)} days | score range: {df[\"sentiment_score\"].min():.3f} to {df[\"sentiment_score\"].max():.3f}')\n",
    "    else:\n",
    "        print(f'WARNING: {path} not found')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## Step 8 — Copy results to Google Drive"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "\n",
    "for ticker in tickers:\n",
    "    src = f'data/sentiment/{ticker}_sentiment.csv'\n",
    "    dst = f'/content/drive/MyDrive/MARL/data/sentiment/{ticker}_sentiment.csv'\n",
    "    shutil.copy(src, dst)\n",
    "    print(f'Copied {ticker}_sentiment.csv → Google Drive')\n",
    "\n",
    "print('\\nAll sentiment files saved to Google Drive.')\n",
    "print('Download them and place in your local data/sentiment/ folder.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 9 — Download files to your laptop\n",
    "Run this cell to download each sentiment CSV directly from Colab to your Downloads folder."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import files\n",
    "\n",
    "for ticker in tickers:\n",
    "    path = f'data/sentiment/{ticker}_sentiment.csv'\n",
    "    files.download(path)\n",
    "    print(f'Downloaded {ticker}_sentiment.csv')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}